# 01 - Quickstart: from a raw product to standardised fields

Every product evaluated in the paper arrives with its own variable names, units
and grid conventions. `alpinemet.io` maps them onto one set, so that every
indicator can be written once.

This notebook builds an ERA5-shaped dataset in memory. It runs with no data, no
credentials and no network; point `open_product` at your own files to do the
same for real.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from alpinemet.io import ALPINE_DOMAIN, DATASETS, get_dataset_spec, standardise

## What the registry knows about each product

In [ ]:
pd.DataFrame(
    [
        {
            "key": spec.key,
            "resolution_km": spec.resolution_km,
            "step_h": spec.timestep_hours,
            "native_gust": spec.has_native_gust,
            "wind_100m": spec.has_100m_wind,
            "grid": spec.grid_type,
        }
        for spec in DATASETS.values()
    ]
).set_index("key").sort_values("resolution_km")

The accumulation convention is **declared** per product rather than inferred
from the numbers. This is the detail that most often goes wrong by a factor of
3600: ERA5 accumulates over each step, ERA5-Land accumulates from a daily reset.

In [ ]:
for key in ["era5", "era5_land", "aifs"]:
    spec = get_dataset_spec(key)
    print(f"{key:<10} solar_radiation -> {spec.accumulation_kind('solar_radiation').value}")

## A synthetic ERA5-shaped product

In [ ]:
time = pd.date_range("2020-06-21", periods=48, freq="h")
lats = np.arange(52.0, 39.9, -1.0)   # ERA5 stores latitude descending
lons = np.arange(0.0, 25.1, 1.0)
dims = ("time", "latitude", "longitude")
coords = {"time": time, "latitude": lats, "longitude": lons}
shape = (time.size, lats.size, lons.size)

diurnal = np.clip(800.0 * np.sin((np.arange(24) - 6) / 12 * np.pi), 0.0, None)
flux = np.tile(diurnal, 2)[:, None, None] * np.ones(shape[1:])

raw = xr.Dataset(
    {
        "t2m": xr.DataArray(
            np.full(shape, 288.15), dims=dims, coords=coords, attrs={"units": "K"}
        ),
        "u10": xr.DataArray(np.full(shape, 3.0), dims=dims, coords=coords),
        "v10": xr.DataArray(np.full(shape, 4.0), dims=dims, coords=coords),
        "ssrd": xr.DataArray(
            flux * 3600.0, dims=dims, coords=coords, attrs={"units": "J m-2"}
        ),
    }
)
raw

## Standardise it

In [ ]:
era5 = standardise(raw, "era5", domain=ALPINE_DOMAIN)
era5

Compare before and after: temperature has moved from kelvin to degrees Celsius,
radiation from accumulated J/m2 to instantaneous W/m2, and a scalar wind speed
has appeared. The descending latitude axis was subset correctly, where a naive
`sel(latitude=slice(43, 49))` would have returned an empty selection with no
error at all.

In [ ]:
print(f"raw  t2m   max: {float(raw['t2m'].max()):>12,.1f} K")
print(f"std  temp  max: {float(era5['temperature_2m'].max()):>12,.1f} degC")
print()
print(f"raw  ssrd  max: {float(raw['ssrd'].max()):>12,.0f} J/m2")
print(f"std  solar max: {float(era5['solar_radiation'].max()):>12,.1f} W/m2")
print()
print(f"derived wind speed: {float(era5['wind_speed_10m'].max()):.1f} m/s")
print(f"latitudes kept: {era5.sizes['latitude']} of {raw.sizes['latitude']}")

Provenance travels with the data, so a file written now can still be explained
later.

In [ ]:
from alpinemet.attrs import decode_attribute

for key in ["dataset_key", "resolution_km", "has_native_gust", "dataset_notes"]:
    print(f"{key:<18} {decode_attribute(era5.attrs[key])}")

## Reading real files

```python
from alpinemet.io import open_product

era5 = open_product(
    "/path/to/DATA/ERA5/era5_2020_AT.nc",
    "era5",
    domain=ALPINE_DOMAIN,
    start="2020-01-01",
    end="2020-12-31",
)
```

Or from the command line, with machine-specific paths kept out of the
repository in `configs/paths.yaml`:

```bash
uv run alpinemet check --config configs/era5_eraland_2020.yaml
uv run alpinemet run   --config configs/era5_eraland_2020.yaml
```

`check` validates the configuration and resolves every input path without
opening a single file, so a typo surfaces in a second rather than after a
multi-hour load.